# backtest_contract_v1

目标：验证 `src/backtest` 的稳定契约，覆盖预测信号转仓位、成本/滑点、walk-forward 切分、analysis 输入和中文 review 证据。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'src').exists():
        ROOT = parent
        break

CARD = ROOT / 'research' / 'backtest_contract_v1'
CARD.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import compare_models, compute_regime_metrics, compute_window_metrics
from src.backtest import (
    BacktestConfig,
    CostModel,
    PositionPolicy,
    build_walk_forward_splits,
    run_prediction_backtest,
)

In [2]:
dates = pd.date_range('2026-01-01', periods=10, freq='D')
predictions = pd.DataFrame({
    'date': list(dates) * 2,
    'model': ['ridge_mse'] * 10 + ['gcn_mse'] * 10,
    'prediction': [0.01, 0.02, -0.01, 0.03, 0.04, -0.03, -0.02, 0.01, 0.02, 0.03]
        + [0.02, 0.04, -0.02, 0.05, 0.05, -0.04, -0.04, 0.02, 0.03, 0.04],
    'target_return': [0.0, 0.02, -0.01, 0.03, 0.04, -0.05, -0.02, 0.01, 0.02, 0.03] * 2,
})
prices = pd.DataFrame({
    'date': dates,
    'close': [100, 102, 101, 104, 108, 102, 100, 101, 103, 106],
    'vix': [18, 19, 22, 25, 28, 38, 35, 30, 24, 20],
})

config = BacktestConfig(
    start='2026-01-01',
    end='2026-01-10',
    benchmark='SOXL',
    position_policy=PositionPolicy(min_position=0.0, max_position=1.0, neutral_position=0.5, lag=1),
    cost_model=CostModel(open_cost_bps=5.0, close_cost_bps=5.0, slippage_bps=2.0),
    stress_windows={'stress_window': ('2026-01-06', '2026-01-07')},
)

result = run_prediction_backtest(predictions, price_frame=prices, config=config)
analysis_frame = result.to_analysis_frame()
regime_metrics = compute_regime_metrics(analysis_frame)
window_metrics = compute_window_metrics(
    analysis_frame,
    windows={'early': ('2026-01-01', '2026-01-05'), 'stress_window': ('2026-01-06', '2026-01-07')},
)
model_comparison = compare_models(analysis_frame)
walk_dates = pd.date_range('2026-01-01', periods=12, freq='D')
walk_forward = pd.DataFrame([split.to_dict() for split in build_walk_forward_splits(walk_dates, train_size=5, test_size=2, step_size=2)])

summary = result.summary_payload()
summary.update({
    'schema_version': 'research.backtest_contract_v1',
    'status': 'completed',
    'daily_rows': int(len(result.daily)),
    'model_count': int(result.daily['model'].nunique()),
    'top_model': str(model_comparison.iloc[0]['model']),
    'top_utility_model': str(result.model_summary.iloc[0]['model']),
    'stress_window_rows': int(result.daily['stress_window'].notna().sum()),
    'cost_return_abs_sum': float(result.daily['cost_return'].abs().sum()),
    'walk_forward_fold_count': int(len(walk_forward)),
    'artifacts': {
        'daily_backtest': 'daily_backtest.csv',
        'model_summary': 'model_summary.csv',
        'model_comparison': 'model_comparison.csv',
        'regime_metrics': 'regime_metrics.csv',
        'window_metrics': 'window_metrics.csv',
        'walk_forward_splits': 'walk_forward_splits.csv',
    },
})

result.daily.to_csv(CARD / 'daily_backtest.csv', index=False)
result.model_summary.to_csv(CARD / 'model_summary.csv', index=False)
model_comparison.to_csv(CARD / 'model_comparison.csv', index=False)
regime_metrics.to_csv(CARD / 'regime_metrics.csv', index=False)
window_metrics.to_csv(CARD / 'window_metrics.csv', index=False)
walk_forward.to_csv(CARD / 'walk_forward_splits.csv', index=False)
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

summary

{'schema_version': 'research.backtest_contract_v1',
 'status': 'completed',
 'config': {'start': '2026-01-01',
  'end': '2026-01-10',
  'benchmark': 'SOXL',
  'position_policy': {'min_position': 0.0,
   'max_position': 1.0,
   'neutral_position': 0.5,
   'signal_scale': None,
   'lag': 1,
   'schema_version': 'backtest.position_policy.v1'},
  'cost_model': {'open_cost_bps': 5.0,
   'close_cost_bps': 5.0,
   'slippage_bps': 2.0,
   'schema_version': 'backtest.cost_model.v1'},
  'stress_windows': {'stress_window': ['2026-01-06', '2026-01-07']},
  'schema_version': 'backtest.config.v1'},
 'model_count': 2,
 'row_count': 20,
 'top_model': 'gcn_mse',
 'daily_rows': 20,
 'top_utility_model': 'gcn_mse',
 'stress_window_rows': 4,
 'cost_return_abs_sum': 0.0033687499999999998,
 'walk_forward_fold_count': 3,
 'artifacts': {'daily_backtest': 'daily_backtest.csv',
  'model_summary': 'model_summary.csv',
  'model_comparison': 'model_comparison.csv',
  'regime_metrics': 'regime_metrics.csv',
  'wind

In [3]:
assert result.schema_version == 'backtest.result.v1'
assert result.config.schema_version == 'backtest.config.v1'
assert summary['cost_return_abs_sum'] > 0.0
assert summary['walk_forward_fold_count'] == 3
assert set(result.daily['stress_window'].dropna()) == {'stress_window'}
assert not regime_metrics.empty
assert not model_comparison.empty
assert (CARD / 'analysis_summary.json').exists()
assert (CARD / 'daily_backtest.csv').exists()

## 中文结论

- 做了什么：新增并验证 `src/backtest` v1 契约，把预测信号转换为滞后执行仓位，并显式计入交易成本和滑点。
- 核心指标：样例包含 2 个模型、20 行日度回测、3 个 walk-forward fold；`daily_backtest.csv`、`model_summary.csv`、`model_comparison.csv`、`regime_metrics.csv`、`window_metrics.csv` 和 `walk_forward_splits.csv` 已写入本研究目录。
- regime 差异：`stress_window` 覆盖 2026-01-06 到 2026-01-07，analysis 可直接对该压力窗口和自动 regime 输出模型差异。
- 是否晋升：晋升为 backtest 框架基础契约；不代表任何交易模型晋升 production。
- 风险：当前样例是合成预测和价格路径，真实 QuantBench 全量模型需要后续接入同一 `BacktestConfig`。
- 下一步：让模型矩阵默认使用 `run_prediction_backtest`，并把成本/滑点、walk-forward fold 写入 dashboard manifest。